In [5]:
def userbased(input_user,ratings,items):
    n_users=ratings.user_id.unique().shape[0]
    n_items=ratings.movie_id.unique().shape[0]
    
    ## Creating Pivot table
    datama=ratings.pivot_table(index='user_id',columns='movie_id',values='rating')
    
    ##Replace Nan with 0
    data_matrix=datama.replace(np.nan,0)
    
    ##Finding Similarity between users & Similarity between items
    from sklearn.metrics.pairwise import pairwise_distances
    user_similarity=pairwise_distances(data_matrix,metric='cosine')
    item_similarity=pairwise_distances(data_matrix.T,metric='cosine')
    
    # Calculate Score value using formula
    def predict(ratings,similarity,type='user'):
        if(type == 'user'):
            mean_user_rating=ratings.mean(axis=1)   ## the average (mean) value across each row is calculated
            #We use np.newaxis so that mean_user_rating hassame format as ratings
            rating_diff=(ratings-mean_user_rating.to_numpy()[:,np.newaxis])
            pred=mean_user_rating.to_numpy()[:,np.newaxis] + similarity.dot(rating_diff)/np.array([np.abs(similarity).sum(axis=1)]).T
        elif type == 'item':
            pred = ratings.dot(similarity)/np.array([np.abs(similarity).sum(axis=1)])
        return pred
    
    #Prediction table
    user_prediction=predict(data_matrix,user_similarity,type='user')
    item_prediction=predict(data_matrix,item_similarity,type='item')

    ## To find the similarity between input user and other users
    ## convert user_similarity into table

    user_sim_table=pd.DataFrame(user_similarity)

    ## Find similar user for input user using cosine table
    similar_input_user=user_sim_table[input_user].sort_values(ascending=True).head(5).index

    #Convert similar_input_user to list
    similar_user_input=list(similar_input_user)

    ## Creating a list of movie_id for similar user
    similar_user_movieid_list=[]
    for sim_user in similar_input_user:
        sim=list(ratings[ratings['user_id']==sim_user]['movie_id'])
        similar_user_movieid_list.append(sim)

    # Convert all the list into single list
    import itertools
    similar_user_movieid_single_list=list(itertools.chain.from_iterable(similar_user_movieid_list))

    #Unique Movie id from the list
    Unique_movieid_similar_user=set(similar_user_movieid_single_list)
    #Input user watched movie list
    input_user_watched_movieid=list(ratings[ratings['user_id']==input_user]['movie_id'].values)

    ## Create a list of movieid for user input
    recomm=[]
    for per_id in Unique_movieid_similar_user:
        if(per_id in input_user_watched_movieid):
            pass
        else:
           recomm.append(per_id)

    ##Filterd movieid have to be checked with prediction table for ratings
    user_pred=pd.DataFrame(user_prediction)
    user_pred_Trans=user_pred.T
    
    ## from the recomm list select the movie that will be liked by the user using prediction table
    highest_Rated=[]
    input_user_pre=pd.DataFrame(user_pred_Trans[input_user])
    input_user_pred=input_user_pre.T
    for re in recomm:
        value=input_user_pred[re].values
        if(value>=1):
            highest_Rated.append(re)
    
    # Creating movie list

    movie_title=[]
    for movieid in highest_Rated:
        mov=items[items['movie id']== movieid]['movie title'].values
        movie_title.append(mov)

    ##Creating movie list in to proper list

    movie_title_list=[]
    for m in movie_title:
        movie_title_list.append(list(m))

    ## convert into single list
    import itertools
    final_recomm_movie_list=list(itertools.chain.from_iterable(movie_title_list))
    print("The common Movie in Recom & User:",list(set(recomm)&set(input_user_watched_movieid)))
    return final_recomm_movie_list

In [13]:
import pandas as pd
import numpy as np

input_user=29

r_cols=['user_id','movie_id','rating','unix_timestamp']
ratings=pd.read_csv("ml-100k/u.data",sep='\t', names=r_cols)  ## can add encoding='latin-1'


# Get the respective movie names for movie id

i_cols = ['movie id', 'movie title' ,'release date','video release date', 'IMDb URL', 'unknown', 'Action', 'Adventure',
'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


items = pd.read_csv('ml-100k/u.item', sep='|', names=i_cols,encoding='latin-1')

final_recomm_movie_list=userbased(input_user,ratings,items)
#print(len(final_recomm_movie_list))
print("Final_recommended_movie_list:\n", final_recomm_movie_list)

The common Movie in Recom & User: []
Final_recommended_movie_list:
 ['Babe (1995)', 'Seven (Se7en) (1995)', 'Postino, Il (1994)', 'Professional, The (1994)', 'Santa Clause, The (1994)', 'Crow, The (1994)', 'Free Willy (1993)', 'Aladdin (1992)', 'Dances with Wolves (1990)', 'Snow White and the Seven Dwarfs (1937)', 'Empire Strikes Back, The (1980)', 'Princess Bride, The (1987)', 'Brazil (1985)', 'Unforgiven (1992)', 'Field of Dreams (1989)', 'Breaking the Waves (1996)', 'Men in Black (1997)', 'Sabrina (1995)', 'Sense and Sensibility (1995)', 'Secrets & Lies (1996)', 'Donnie Brasco (1997)', 'In & Out (1997)', 'Client, The (1994)', 'Pinocchio (1940)']
